# Projet Machine Learning — Prédiction du diabète

**Auteur :** Emna  
**Encadrant :** M. Abdallah Khemais  
**Module :** Machine Learning / Deep Learning

---

## 1. Contexte et objectif

Ce projet vise à prédire la présence ou l'absence de diabète à partir de mesures cliniques. Il s'inscrit dans une problématique de **classification médicale**, domaine couvert par les sources Stanford en santé, avec un dataset issu de l'**UCI Machine Learning Repository** (Source 3).

## 2. Dataset

| Propriété | Valeur |
|-----------|--------|
| Source | [UCI Pima Indians Diabetes](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database) |
| Lignes | 768 |
| Features | 8 variables cliniques + features dérivées |
| Cible | `Outcome` (0 = non diabétique, 1 = diabétique) |

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from preprocessing import (
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    add_engineered_features,
    build_preprocessor,
    get_feature_columns,
    load_data,
)
from train import evaluate_model, get_models, save_best_model

sns.set_theme(style='whitegrid')
df = add_engineered_features(load_data(ROOT / 'data' / 'diabetes.csv'))
df[FEATURE_COLUMNS + [TARGET_COLUMN]].describe().T

## 3. Analyse exploratoire (synthèse)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x=TARGET_COLUMN, ax=axes[0])
axes[0].set_title('Classes déséquilibrées (~65% / 35%)')
sns.heatmap(df[FEATURE_COLUMNS + [TARGET_COLUMN]].corr(), annot=True, fmt='.2f', ax=axes[1])
axes[1].set_title('Corrélation — Glucose fortement lié à Outcome')
plt.tight_layout()
plt.show()

## 4. Pipeline de preprocessing

1. Remplacement des zéros impossibles (Glucose, BMI, etc.) par `NaN`
2. Imputation (médiane / mode)
3. Standardisation des variables numériques
4. One-Hot Encoding des catégories dérivées (`AgeGroup`, `BMICategory`)
5. Split stratifié 80/20

In [ ]:
feature_columns = get_feature_columns(include_engineered=True)
X = df[feature_columns]
y = df[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
preprocessor = build_preprocessor(feature_columns)

## 5. Comparaison des modèles

In [ ]:
results = []
fitted = {}
for name, model in get_models().items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    metrics = evaluate_model(y_test, y_pred, y_proba)
    metrics['cv_f1_mean'] = cross_val_score(pipe, X, y, cv=5, scoring='f1').mean()
    metrics['model'] = name
    results.append(metrics)
    fitted[name] = pipe

results_df = pd.DataFrame(results).set_index('model').round(4)
results_df.sort_values('f1', ascending=False)

## 6. Modèle retenu et interprétation

In [ ]:
best_name = results_df['f1'].idxmax()
best_model = fitted[best_name]
save_best_model(best_model, ROOT / 'models' / 'best_model.joblib')
print(classification_report(y_test, best_model.predict(X_test)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, ax=axes[0])
RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=axes[1])
plt.suptitle(f'Modèle retenu : {best_name}')
plt.tight_layout()
plt.show()

## 7. Conclusion

- **Random Forest** obtient le meilleur compromis F1 / rappel sur le jeu de test.
- La glycémie et l'IMC restent les variables les plus discriminantes.
- **Limites :** population spécifique (Pima Indians), dataset de petite taille, données manquantes codées en zéros.
- **Pistes :** SMOTE pour le déséquilibre, modèles calibrés, validation externe.